In [ ]:
import pandas as pd

books = pd.read_csv("books_categories.csv")

In [ ]:
from transformers import pipeline
classifier = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base", top_k = None, device="cuda")
classifier("I am so happy today!")

In [ ]:
books["description"][0]

In [ ]:
sentences = books["description"][0].split(".")
predictions = classifier(sentences)

In [ ]:
sorted(predictions[0], key=lambda x: x['label'])

In [ ]:
import numpy as np

emotion_labels =  ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"]
isbn = []
emotion_scores = {label: [] for label in emotion_labels}

def calculate_max_emotion_scores(predictions):
    per_emotion_scores = {label: [] for label in emotion_labels}
    for prediction in predictions:
        sorted_predictions = sorted(prediction, key=lambda x: x['label'])
        for index, label in enumerate(emotion_labels):
            per_emotion_scores[label].append(sorted_predictions[index]['score'])
    return {label: np.max(scores) for label, scores in per_emotion_scores.items()}

In [ ]:
from tqdm import tqdm

emotion_labels =  ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"]
isbn = []
emotion_scores = {label: [] for label in emotion_labels}

for i in tqdm(range(len(books))):
    isbn.append(books["isbn13"][i])
    sentences = books["description"][i].split(".")
    predictions = classifier(sentences)
    max_scores = calculate_max_emotion_scores(predictions)
    for label in emotion_labels:
        emotion_scores[label].append(max_scores[label])

In [ ]:
emotion_df = pd.DataFrame(emotion_scores)
emotion_df["isbn13"] = isbn

In [ ]:
emotion_df.head()

In [ ]:
books = pd.merge(books, emotion_df, on="isbn13")

In [ ]:
books.to_csv("books_with_emotions.csv", index=False)